In [16]:
import math

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

from rdkit.Chem import PandasTools

In [17]:
bioactivities_df = pd.read_csv(r"E:\Projects\Mycobacterium tuberculosis\01-Datadownload\input\Activities.csv",encoding='gbk')

In [18]:
bioactivities_df

,Molecule ChEMBL ID,Smiles,Standard Type,Standard Relation,Standard Value,Standard Units
0,CHEMBL376488,COc1nc2ccc(Br)cc2cc1[C@@H](c1ccccc1)[C@@](O)(C...,log10CFU/ml,'=',-0.37,NaN
1,CHEMBL374478,CO[C@H]1/C=C/O[C@@]2(C)Oc3c(C)c(O)c4c(O)c(c(/C...,log10CFU/ml,'=',-1.70,NaN
2,CHEMBL1287829,O=C(N/N=C1\C2CCCC1[C@H](c1ccc(Cl)cc1)N[C@@H]2c...,Activity,'=',63.61,%
3,CHEMBL1289484,O=C(N/N=C1\C2CCCC1[C@H](c1ccccc1)N[C@@H]2c1ccc...,Activity,'=',77.52,%
4,CHEMBL374478,CO[C@H]1/C=C/O[C@@]2(C)Oc3c(C)c(O)c4c(O)c(c(/C...,MIC99,'=',1.00,ug ml-1
...,...,...,...,...,...,...
208228,CHEMBL5558043,O=C1C(Cl)=C(N2C(=O)C(Cl)C2c2ccccc2)C(=O)c2ccccc21,MIC,'=',6.25,ug.mL-1
208229,CHEMBL5558772,COc1ccc(C2C(Cl)C(=O)N2C2=C(Cl)C(=O)c3ccccc3C2=...,MIC,'=',31070.00,nM
208230,CHEMBL5555020,O=C1C(Cl)=C(N2C(=O)C(Cl)C2c2ccc(Cl)cc2)C(=O)c2...,MIC99.9,'=',6.25,ug ml-1
208231,CHEMBL5555799,CN(C)c1ccc(C2C(Cl)C(=O)N2C2=C(Cl)C(=O)c3ccccc3...,MIC99.9,'=',6.25,ug ml-1


In [19]:
bioactivities_df = bioactivities_df.astype({"Standard Value": "float64"})

In [20]:
bioactivities_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 208233 entries, 0 to 208232
Data columns (total 6 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Molecule ChEMBL ID  208233 non-null  object 
 1   Smiles              205184 non-null  object 
 2   Standard Type       205743 non-null  object 
 3   Standard Relation   200118 non-null  object 
 4   Standard Value      200114 non-null  float64
 5   Standard Units      196504 non-null  object 
dtypes: float64(1), object(5)
memory usage: 9.5+ MB


In [21]:
print(bioactivities_df.isnull().sum())

Molecule ChEMBL ID        0
Smiles                 3049
Standard Type          2490
Standard Relation      8115
Standard Value         8119
Standard Units        11729
dtype: int64


In [22]:
bioactivities_df = bioactivities_df.dropna(subset=["Smiles", "Standard Value"])
print(f"DataFrame shape: {bioactivities_df.shape}")

DataFrame shape: (199555, 6)


In [23]:
bioactivities_df

,Molecule ChEMBL ID,Smiles,Standard Type,Standard Relation,Standard Value,Standard Units
0,CHEMBL376488,COc1nc2ccc(Br)cc2cc1[C@@H](c1ccccc1)[C@@](O)(C...,log10CFU/ml,'=',-0.37,NaN
1,CHEMBL374478,CO[C@H]1/C=C/O[C@@]2(C)Oc3c(C)c(O)c4c(O)c(c(/C...,log10CFU/ml,'=',-1.70,NaN
2,CHEMBL1287829,O=C(N/N=C1\C2CCCC1[C@H](c1ccc(Cl)cc1)N[C@@H]2c...,Activity,'=',63.61,%
3,CHEMBL1289484,O=C(N/N=C1\C2CCCC1[C@H](c1ccccc1)N[C@@H]2c1ccc...,Activity,'=',77.52,%
4,CHEMBL374478,CO[C@H]1/C=C/O[C@@]2(C)Oc3c(C)c(O)c4c(O)c(c(/C...,MIC99,'=',1.00,ug ml-1
...,...,...,...,...,...,...
208228,CHEMBL5558043,O=C1C(Cl)=C(N2C(=O)C(Cl)C2c2ccccc2)C(=O)c2ccccc21,MIC,'=',6.25,ug.mL-1
208229,CHEMBL5558772,COc1ccc(C2C(Cl)C(=O)N2C2=C(Cl)C(=O)c3ccccc3C2=...,MIC,'=',31070.00,nM
208230,CHEMBL5555020,O=C1C(Cl)=C(N2C(=O)C(Cl)C2c2ccc(Cl)cc2)C(=O)c2...,MIC99.9,'=',6.25,ug ml-1
208231,CHEMBL5555799,CN(C)c1ccc(C2C(Cl)C(=O)N2C2=C(Cl)C(=O)c3ccccc3...,MIC99.9,'=',6.25,ug ml-1


In [24]:
print(bioactivities_df.isnull().sum())

Molecule ChEMBL ID       0
Smiles                   0
Standard Type            0
Standard Relation        1
Standard Value           0
Standard Units        4038
dtype: int64


In [25]:
print(f"Units in downloaded data: {bioactivities_df['Standard Units'].unique()}")

Units in downloaded data: [nan '%' 'ug ml-1' 'ug.mL-1' 'uM' 'nM' 'log10CFU' '10^3/ml' '10^4/ml'
 "10'7/g" 'CFU' 'mg/ml' 'mm' 'mg kg-1' 'mg' 'RLU' 'log CFU' '/ml'
 '10^-6No_unit' '10^5/ml' 'day' 'log10CFU/ml' '10^2/ml' 'mg/L'
 'deltalog10CFU' 'mM' '/uL' 'ng/ml' 'deltalog10CFU/ml' 'microM' 'umol/cm3'
 "10'7CFU" '10^-7No_unit' "10'3CFU" 'ng.hr.mL-1' '10^8CFU/ml'
 "10'-3 log10CFU/ml/hr" 'log10RLU' "10'5CFU" '10^-5No_unit' 'microg/ml'
 '10^9CFU' '10^-8No_unit']


In [26]:
bioactivities_df = bioactivities_df[bioactivities_df["Standard Units"].isin(["ug.mL-1","ug ml-1","mg/L","microg/ml","microg.mL-1","μg/mL","μg·mL⁻¹","mg/L"])]
print(f"Units after filtering: {bioactivities_df['Standard Units'].unique()}")
print(f"DataFrame shape: {bioactivities_df.shape}")

Units after filtering: ['ug ml-1' 'ug.mL-1' 'mg/L' 'microg/ml']
DataFrame shape: (30167, 6)


In [27]:
bioactivities_df

,Molecule ChEMBL ID,Smiles,Standard Type,Standard Relation,Standard Value,Standard Units
4,CHEMBL374478,CO[C@H]1/C=C/O[C@@]2(C)Oc3c(C)c(O)c4c(O)c(c(/C...,MIC99,'=',1.000,ug ml-1
6,CHEMBL1086385,CCCCCCCC[C@@H](C)C(=O)N1CCC[C@H]1C(=O)N[C@@H](...,MIC,'=',0.120,ug.mL-1
7,CHEMBL1086383,CCCCCCCC[C@@H](C)C(=O)N1CCC[C@H]1C(=O)N[C@@H](...,MIC,'=',2.000,ug.mL-1
14,CHEMBL225164,Oc1ccc(Cl)c2cccnc12,MIC,'=',0.125,ug.mL-1
15,CHEMBL497,Oc1c(I)cc(Cl)c2cccnc12,MIC,'=',6.250,ug.mL-1
...,...,...,...,...,...,...
208226,CHEMBL5574294,O=C(CSc1nc2ccc(Br)cc2s1)NC1CCCCC1,MIC90,'=',64.000,ug.mL-1
208227,CHEMBL1966714,O=C(CSc1nc2ccccc2c(=O)[nH]1)NC1CCCCC1,IC50,'=',2.226,ug.mL-1
208228,CHEMBL5558043,O=C1C(Cl)=C(N2C(=O)C(Cl)C2c2ccccc2)C(=O)c2ccccc21,MIC,'=',6.250,ug.mL-1
208230,CHEMBL5555020,O=C1C(Cl)=C(N2C(=O)C(Cl)C2c2ccc(Cl)cc2)C(=O)c2...,MIC99.9,'=',6.250,ug ml-1


In [28]:
# bioactivities_df.drop_duplicates("Molecule ChEMBL ID", keep="first", inplace=True)
bioactivities_df = bioactivities_df.drop_duplicates("Molecule ChEMBL ID", keep="first")
print(bioactivities_df.shape)

(16557, 6)


In [29]:
print(f"Type in downloaded data: {bioactivities_df['Standard Type'].unique()}")

Type in downloaded data: ['MIC99' 'MIC' 'MIC90' 'IC90' 'MIC50' 'MIC=>90' 'IC50' 'MIC=>80' 'MBC'
 'EC99' 'TBMIC' 'Activity' 'Concentration' 'MIC>90' 'MIC=<90' 'MIC95'
 'EC90' 'MBC90' 'GI99' 'GI50' 'INH' 'IC' 'FIC' 'GI' 'MIC99.9' 'GI90'
 'Ratio' 'MIC100' 'IC99' 'MBC99.9' 'MIC>99']


In [30]:
bioactivities_df = bioactivities_df[bioactivities_df["Standard Type"] == "MIC"]
print(f"DataFrame shape: {bioactivities_df.shape}")

DataFrame shape: (12756, 6)


In [31]:
#Since we deleted some rows, but we want to iterate over the index later, we reset the index to be continuous.
bioactivities_df.reset_index(drop=True, inplace=True)

In [32]:
bioactivities_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12756 entries, 0 to 12755
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Molecule ChEMBL ID  12756 non-null  object 
 1   Smiles              12756 non-null  object 
 2   Standard Type       12756 non-null  object 
 3   Standard Relation   12755 non-null  object 
 4   Standard Value      12756 non-null  float64
 5   Standard Units      12756 non-null  object 
dtypes: float64(1), object(5)
memory usage: 598.1+ KB


In [33]:
bioactivities_df.rename(
    columns={"Standard Value": "MIC", "Standard Units": "Units"}, inplace=True
)
bioactivities_df.shape

(12756, 6)

In [34]:
bioactivities_df = bioactivities_df.drop(columns=['Standard Relation'])

In [35]:
bioactivities_df

,Molecule ChEMBL ID,Smiles,Standard Type,MIC,Units
0,CHEMBL1086385,CCCCCCCC[C@@H](C)C(=O)N1CCC[C@H]1C(=O)N[C@@H](...,MIC,0.120,ug.mL-1
1,CHEMBL1086383,CCCCCCCC[C@@H](C)C(=O)N1CCC[C@H]1C(=O)N[C@@H](...,MIC,2.000,ug.mL-1
2,CHEMBL225164,Oc1ccc(Cl)c2cccnc12,MIC,0.125,ug.mL-1
3,CHEMBL497,Oc1c(I)cc(Cl)c2cccnc12,MIC,6.250,ug.mL-1
4,CHEMBL4061841,NC(=O)C1CCN(Cc2c(-c3ccccc3)[nH]c3ccnc(Cl)c23)CC1,MIC,25.000,ug.mL-1
...,...,...,...,...,...
12751,CHEMBL5517699,CCOc1cc(/C=N/C2=C(Cl)C(=O)c3ccccc3C2=O)ccc1O,MIC,6.250,ug.mL-1
12752,CHEMBL5572792,COc1cc([C@@](O)(CCN(C)C)[C@H](c2cc3cc(Br)ccc3n...,MIC,0.004,ug.mL-1
12753,CHEMBL4854563,CO[C@H]1CC=C2C(=O)c3c(cc(C)c(-c4c(C)cc5c(c4O)C...,MIC,25.000,ug.mL-1
12754,CHEMBL4864131,CCOC(=O)NC(=O)c1ccsc1NC(=O)c1ccc(C(=O)N2CCOCC2...,MIC,1.660,ug.mL-1


In [36]:
print(bioactivities_df.isnull().sum())

Molecule ChEMBL ID    0
Smiles                0
Standard Type         0
MIC                   0
Units                 0
dtype: int64


In [37]:
bioactivities_df.to_csv(r"E:\Projects\Mycobacterium tuberculosis\01-Datadownload\bio_12756.csv",index=False)